# Quantum Resource Ablation & Classical-Simulability Scaling

**ETH Quantum Hackathon 2026 â€” Quandela Challenge**

**Objective:** Test whether the MerLin photonic layer is a meaningful quantum resource or merely a classically simulable feature map.

This notebook focuses on two questions:
1. **Which photonic/quantum resource is responsible for MerLin model performance?**
2. **Is the MerLin model operating in a regime that becomes hard to simulate classically?**

Approach:
- **Quantum-resource ablation:** Surgical and retrained ablations of mode mixing, trainability, optical phases, circuit depth, photon distinguishability, and photon number.
- **Classical-simulability scaling:** Controlled sweeps of depth, modes, and photons to measure empirical simulation cost (Fock dimension, forward time, training time).

**Constraints:**
- `FAST_MODE` available for quick validation.
- Every unsupported experiment fails gracefully with `supported=False` and a clear reason.
- All results are saved to `results/quantum_resource_ablation_scaling_results.csv`.


In [ ]:
import matplotlib
matplotlib.use("Agg")

import math, time, random, os, warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple
from contextlib import contextmanager
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

import merlin as ML
import perceval as pcvl

DEVICE = torch.device("cpu")
DTYPE = torch.float32

CONFIG = {
    "FAST_MODE": True,
    "seeds": [0],
    "base_epochs": 50,
    "sweep_epochs": 50,
    "lr": 1e-2,
    "n_f": 64,
    "n_i": 64,
    "n_b": 64,
    "nx_eval": 51,
    "nt_eval": 51,
    "feature_size": 4,
    "quantum_output_size": 4,
    "hidden": 16,
    "baseline_depth": 1,
    "baseline_photons": 2,
    "depth_values_fast": [0, 1, 2],
    "depth_values_full": [0, 1, 2, 3, 4],
    "mode_values_fast": [2, 3, 4],
    "mode_values_full": [2, 3, 4, 5, 6, 8],
    "photon_values_fast": [1, 2],
    "photon_values_full": [1, 2, 3],
    "mu_values": [1.0, 0.75, 0.5, 0.25, 0.0],
    "output_dir": "results",
}

if not CONFIG["FAST_MODE"]:
    CONFIG["base_epochs"] = 300
    CONFIG["sweep_epochs"] = 150
    CONFIG["seeds"] = [0, 1, 2]
    CONFIG["nx_eval"] = 101
    CONFIG["nt_eval"] = 101

print(f"Device: {DEVICE}")
print(f"Torch:  {torch.__version__}")
print(f"MerLin: {getattr(ML, '__version__', 'unknown')}")
print(f"FAST_MODE: {CONFIG['FAST_MODE']}")

os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(CONFIG["output_dir"] + "/figures", exist_ok=True)


In [ ]:
def set_seed(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

@contextmanager
def timer():
    start = time.time()
    elapsed = [0.0]
    try:
        yield elapsed
    finally:
        elapsed[0] = time.time() - start

def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def fock_dimension(modes: int, photons: int) -> int:
    from math import comb
    return comb(modes + photons - 1, photons)

def collision_free_dimension(modes: int, photons: int) -> int:
    from math import comb
    return comb(modes, photons)

def count_components(model: nn.Module) -> Tuple[int, int, int]:
    ql = None
    for m in model.modules():
        if hasattr(m, 'quantum_layer'):
            ql = m.quantum_layer
            break
    if ql is None and hasattr(model, 'quantum'):
        qmod = model.quantum
        if hasattr(qmod, 'quantum_layer'):
            ql = qmod.quantum_layer
    if ql is None:
        return 0, 0, 0
    circ = getattr(ql, 'circuit', None)
    if circ is None:
        return 0, 0, 0
    try:
        ncomp = circ.ncomponents()
        desc = circ.describe()
        bs = desc.count('BS.')
        ps = desc.count('PS(')
        return bs, ps, ncomp
    except Exception:
        return 0, 0, 0

def compute_metrics(pred: np.ndarray, ref: np.ndarray) -> Dict[str, float]:
    diff = pred - ref
    rmse = float(np.sqrt(np.mean(diff**2)))
    nmse = float(np.mean(diff**2) / (np.mean(ref**2) + 1e-12))
    mae = float(np.mean(np.abs(diff)))
    max_err = float(np.max(np.abs(diff)))
    return {"rmse": rmse, "nmse": nmse, "mae": mae, "max_err": max_err}


In [ ]:
ALPHA = 0.30
X_MIN, X_MAX = -math.pi/2, math.pi/2
T_MIN, T_MAX = 0.0, 0.5
SIGMA2 = 0.2
X0 = -math.pi/8

def initial_condition(x: np.ndarray) -> np.ndarray:
    return 0.5 * np.exp(-(x - X0)**2 / (2 * SIGMA2))

def solve_reference(nx=401, nt_eval=401, t_span=(T_MIN, T_MAX)):
    x = np.linspace(X_MIN, X_MAX, nx)
    dx = x[1] - x[0]
    u0 = initial_condition(x)
    u0_interior = u0[1:-1].astype(np.float64)
    n = nx - 2
    main = -2.0 * np.ones(n)
    off = 1.0 * np.ones(n - 1)
    L = (np.diag(main) + np.diag(off, 1) + np.diag(off, -1)) / (dx**2)
    def rhs(t, u):
        return ALPHA * (L @ u)
    t_eval = np.linspace(t_span[0], t_span[1], nt_eval)
    sol = solve_ivp(rhs, t_span, u0_interior, t_eval=t_eval, method='RK45')
    U = np.zeros((nx, nt_eval))
    U[1:-1, :] = sol.y
    U[0, :] = 0.0
    U[-1, :] = 0.0
    X_grid, T_grid = np.meshgrid(x, t_eval, indexing='ij')
    return X_grid, T_grid, U

print("Computing reference solution...")
X_ref, T_ref, U_ref = solve_reference(nx=401, nt_eval=401)
print(f"Reference grid: {X_ref.shape}, range [{U_ref.min():.4f}, {U_ref.max():.4f}]")

from scipy.interpolate import RegularGridInterpolator

def reference_at(x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    x_np = x.detach().cpu().numpy().ravel()
    t_np = t.detach().cpu().numpy().ravel()
    x_vec = np.linspace(X_MIN, X_MAX, X_ref.shape[0])
    t_vec = np.linspace(T_MIN, T_MAX, X_ref.shape[1])
    interp = RegularGridInterpolator((x_vec, t_vec), U_ref, bounds_error=False, fill_value=0.0)
    vals = interp(np.stack([x_np, t_np], axis=1))
    return torch.tensor(vals, dtype=DTYPE, device=DEVICE).view(-1, 1)

def gradients(y: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    return torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y), create_graph=True)[0]

def bc_envelope(x: torch.Tensor) -> torch.Tensor:
    a = math.pi / 2
    return (a**2 - x**2)

def bc_envelope_d1(x: torch.Tensor) -> torch.Tensor:
    return -2.0 * x

def bc_envelope_d2(x: torch.Tensor) -> torch.Tensor:
    return torch.full_like(x, -2.0)


In [ ]:
class ClassicalAuxPINN(nn.Module):
    def __init__(self, hidden: int = 32, depth: int = 3):
        super().__init__()
        layers = [nn.Linear(2, hidden), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden, hidden), nn.Tanh()]
        layers.append(nn.Linear(hidden, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, xt):
        out = self.net(xt)
        q = out[:, 0:1]
        qx_hat = out[:, 1:2]
        x = xt[:, 0:1]
        env = bc_envelope(x)
        env_d1 = bc_envelope_d1(x)
        T = env * q
        Tx = env_d1 * q + env * qx_hat
        return T, Tx

class MerlinAuxQPINN(nn.Module):
    def __init__(self, feature_size=4, quantum_output_size=4, hidden=16, depth=1, n_photons=3):
        super().__init__()
        self.feature_map = nn.Sequential(
            nn.Linear(2, hidden), nn.Tanh(),
            nn.Linear(hidden, feature_size),
        )
        self.n_photons = n_photons
        self.depth = depth
        if depth == 0:
            self.quantum = nn.Linear(feature_size, quantum_output_size)
            self.post = nn.Identity()
        elif depth == 1:
            self.quantum = ML.QuantumLayer.simple(
                input_size=feature_size, output_size=quantum_output_size)
            self.post = nn.Identity()
        else:
            n_modes = max(feature_size + 1, n_photons)
            builder = ML.CircuitBuilder(n_modes=n_modes)
            for i in range(depth):
                builder.add_entangling_layer(trainable=True, name=f"L{i}")
            builder.add_angle_encoding(modes=list(range(feature_size)), name="input")
            for i in range(depth):
                builder.add_entangling_layer(trainable=True, name=f"R{i}")
            circ = builder.to_pcvl_circuit()
            exp = pcvl.Experiment(m_circuit=circ)
            state_list = [1 if idx < n_photons else 0 for idx in range(n_modes)]
            exp.with_input(pcvl.BasicState(state_list))
            self.quantum = ML.QuantumLayer(
                experiment=exp, n_photons=n_photons,
                trainable_parameters=[f"L{i}" for i in range(depth)] + [f"R{i}" for i in range(depth)],
                input_parameters=["input"],
                measurement_strategy=ML.MeasurementStrategy.probs(),
            )
            q_out = self.quantum.output_size
            if q_out != quantum_output_size:
                self.post = ML.ModGrouping(input_size=q_out, output_size=quantum_output_size)
            else:
                self.post = nn.Identity()
        self.readout = nn.Sequential(
            nn.Linear(quantum_output_size, hidden), nn.Tanh(),
            nn.Linear(hidden, 2),
        )

    def forward(self, xt):
        x = xt[:, 0:1]
        z = self.feature_map(xt)
        q = self.quantum(z)
        if self.depth > 1:
            q = self.post(q)
        out = self.readout(q)
        q_u = out[:, 0:1]
        qx_hat = out[:, 1:2]
        env = bc_envelope(x)
        env_d1 = bc_envelope_d1(x)
        T = env * q_u
        Tx = env_d1 * q_u + env * qx_hat
        return T, Tx

class MerlinNoMixingAuxQPINN(nn.Module):
    def __init__(self, feature_size=4, quantum_output_size=4, hidden=16, depth=1, n_photons=3):
        super().__init__()
        self.feature_map = nn.Sequential(
            nn.Linear(2, hidden), nn.Tanh(),
            nn.Linear(hidden, feature_size),
        )
        self.n_photons = n_photons
        self.depth = depth
        n_modes = max(feature_size + 1, n_photons)
        circ = pcvl.Circuit(m=n_modes)
        for d in range(depth):
            for i in range(feature_size):
                circ.add(i, pcvl.PS(pcvl.P(f"phi_pre_d{d}_m{i}")))
        for i in range(feature_size):
            circ.add(i, pcvl.PS(pcvl.P(f"input{i+1}")))
        for d in range(depth):
            for i in range(feature_size):
                circ.add(i, pcvl.PS(pcvl.P(f"phi_post_d{d}_m{i}")))
        exp = pcvl.Experiment(m_circuit=circ)
        state_list = [1 if idx < n_photons else 0 for idx in range(n_modes)]
        exp.with_input(pcvl.BasicState(state_list))
        trainable_names = []
        for d in range(depth):
            for i in range(feature_size):
                trainable_names.append(f"phi_pre_d{d}_m{i}")
                trainable_names.append(f"phi_post_d{d}_m{i}")
        input_names = [f"input{i+1}" for i in range(feature_size)]
        self.quantum = ML.QuantumLayer(
            experiment=exp, n_photons=n_photons,
            trainable_parameters=trainable_names,
            input_parameters=input_names,
            measurement_strategy=ML.MeasurementStrategy.probs(),
        )
        q_out = self.quantum.output_size
        if q_out != quantum_output_size:
            self.post = ML.ModGrouping(input_size=q_out, output_size=quantum_output_size)
        else:
            self.post = nn.Identity()
        self.readout = nn.Sequential(
            nn.Linear(quantum_output_size, hidden), nn.Tanh(),
            nn.Linear(hidden, 2),
        )

    def forward(self, xt):
        x = xt[:, 0:1]
        z = self.feature_map(xt)
        q = self.quantum(z)
        q = self.post(q)
        out = self.readout(q)
        q_u = out[:, 0:1]
        qx_hat = out[:, 1:2]
        env = bc_envelope(x)
        env_d1 = bc_envelope_d1(x)
        T = env * q_u
        Tx = env_d1 * q_u + env * qx_hat
        return T, Tx

class ClassicalFourierPINN(nn.Module):
    def __init__(self, hidden=32, depth=3, n_fourier=16):
        super().__init__()
        self.B = nn.Parameter(torch.randn(2, n_fourier) * 2 * math.pi, requires_grad=False)
        layers = [nn.Linear(n_fourier * 2, hidden), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden, hidden), nn.Tanh()]
        layers.append(nn.Linear(hidden, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, xt):
        x = xt[:, 0:1]
        z = torch.cat([torch.sin(xt @ self.B), torch.cos(xt @ self.B)], dim=1)
        out = self.net(z)
        q = out[:, 0:1]
        qx_hat = out[:, 1:2]
        env = bc_envelope(x)
        env_d1 = bc_envelope_d1(x)
        T = env * q
        Tx = env_d1 * q + env * qx_hat
        return T, Tx



In [ ]:
@dataclass
class TrainConfig:
    epochs: int = 50
    n_f: int = 64
    n_i: int = 64
    n_b: int = 64
    lr: float = 1e-2
    lambda_pde: float = 1.0
    lambda_ic: float = 10.0
    lambda_bc: float = 1.0
    lambda_consistency: float = 0.1
    print_every: int = 0

def sample_interior(n, device=DEVICE, dtype=DTYPE):
    x = torch.empty(n, 1, device=device, dtype=dtype).uniform_(X_MIN, X_MAX)
    t = torch.empty(n, 1, device=device, dtype=dtype).uniform_(T_MIN, T_MAX)
    return torch.cat([x, t], dim=1)

def sample_initial(n, device=DEVICE, dtype=DTYPE):
    x = torch.empty(n, 1, device=device, dtype=dtype).uniform_(X_MIN, X_MAX)
    t = torch.full((n, 1), T_MIN, device=device, dtype=dtype)
    return torch.cat([x, t], dim=1), reference_at(x, torch.zeros_like(x))

def sample_boundary(n, device=DEVICE, dtype=DTYPE):
    side = torch.rand(n, 1, device=device, dtype=dtype)
    x = torch.where(side < 0.5, torch.full_like(side, X_MIN), torch.full_like(side, X_MAX))
    t = torch.empty(n, 1, device=device, dtype=dtype).uniform_(T_MIN, T_MAX)
    return torch.cat([x, t], dim=1), torch.zeros(n, 1, device=device, dtype=dtype)

def pde_residual_aux(model, xt):
    xt = xt.requires_grad_(True)
    T, Tx = model(xt)
    T_t = gradients(T, xt)[:, 1:2]
    Tx_x = gradients(Tx, xt)[:, 0:1]
    r_f = T_t - ALPHA * Tx_x
    T_x_auto = gradients(T, xt)[:, 0:1]
    r_c = T_x_auto - Tx
    return r_f, r_c

def train_model(model, config, print_every=None):
    if print_every is None:
        print_every = config.print_every
    model = model.to(device=DEVICE, dtype=DTYPE)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
    mse = nn.MSELoss()
    history = {"total": [], "pde": [], "ic": [], "bc": [], "consistency": []}
    start = time.time()
    for epoch in range(1, config.epochs + 1):
        optimizer.zero_grad()
        xt_f = sample_interior(config.n_f)
        xt_i, y_i = sample_initial(config.n_i)
        xt_b, y_b = sample_boundary(config.n_b)
        r_f, r_c = pde_residual_aux(model, xt_f)
        loss_pde = mse(r_f, torch.zeros_like(r_f))
        loss_cons = mse(r_c, torch.zeros_like(r_c))
        T_i, _ = model(xt_i)
        T_b, _ = model(xt_b)
        loss_ic = mse(T_i, y_i)
        loss_bc = mse(T_b, y_b)
        loss = (config.lambda_pde * loss_pde + config.lambda_ic * loss_ic
                + config.lambda_bc * loss_bc + config.lambda_consistency * loss_cons)
        loss.backward()
        optimizer.step()
        history["total"].append(loss.item())
        history["pde"].append(loss_pde.item())
        history["ic"].append(loss_ic.item())
        history["bc"].append(loss_bc.item())
        history["consistency"].append(loss_cons.item())
        if print_every > 0 and (epoch % print_every == 0 or epoch == 1):
            print(f"Epoch {epoch:4d} | loss={loss.item():.3e} | pde={loss_pde.item():.3e} | "
                  f"ic={loss_ic.item():.3e} | bc={loss_bc.item():.3e} | cons={loss_cons.item():.3e}")
    elapsed = time.time() - start
    return history, elapsed

def evaluate_model(model, nx=None, nt=None, t_span=(T_MIN, T_MAX)):
    if nx is None: nx = CONFIG["nx_eval"]
    if nt is None: nt = CONFIG["nt_eval"]
    model.eval()
    with torch.no_grad():
        x = torch.linspace(X_MIN, X_MAX, nx, device=DEVICE, dtype=DTYPE).view(-1, 1)
        t = torch.linspace(t_span[0], t_span[1], nt, device=DEVICE, dtype=DTYPE).view(-1, 1)
        Xmesh, Tmesh = torch.meshgrid(x.squeeze(), t.squeeze(), indexing="ij")
        xt = torch.stack([Xmesh.reshape(-1), Tmesh.reshape(-1)], dim=1)
        T_pred, _ = model(xt)
        T_pred = T_pred.reshape(nx, nt).cpu().numpy()
    x_vec = np.linspace(X_MIN, X_MAX, X_ref.shape[0])
    t_vec = np.linspace(T_MIN, T_MAX, X_ref.shape[1])
    interp = RegularGridInterpolator((x_vec, t_vec), U_ref, bounds_error=False, fill_value=0.0)
    X_np = Xmesh.cpu().numpy()
    T_np = Tmesh.cpu().numpy()
    T_ref_grid = interp(np.stack([X_np.ravel(), T_np.ravel()], axis=1)).reshape(nx, nt)
    metrics = compute_metrics(T_pred, T_ref_grid)
    xi = torch.rand(min(2000, nx*nt), 1, device=DEVICE, dtype=DTYPE).uniform_(X_MIN, X_MAX)
    ti = torch.rand(min(2000, nx*nt), 1, device=DEVICE, dtype=DTYPE).uniform_(t_span[0], t_span[1])
    xti = torch.cat([xi, ti], dim=1)
    xti.requires_grad_(True)
    r_f, _ = pde_residual_aux(model, xti)
    pde_mse = torch.mean(r_f**2).item()
    xt_ic, y_ic = sample_initial(500)
    T_ic, _ = model(xt_ic)
    ic_mse = torch.mean((T_ic - y_ic)**2).item()
    xt_bc, y_bc = sample_boundary(500)
    T_bc, _ = model(xt_bc)
    bc_mse = torch.mean((T_bc - y_bc)**2).item()
    xti2 = torch.cat([xi, ti], dim=1)
    xti2.requires_grad_(True)
    _, r_c = pde_residual_aux(model, xti2)
    cons_mse = torch.mean(r_c**2).item()
    dummy = torch.randn(64, 2, device=DEVICE, dtype=DTYPE)
    t0 = time.time()
    with torch.no_grad():
        for _ in range(10):
            _ = model(dummy)
    forward_time_ms = (time.time() - t0) / 10 * 1000
    return {
        **metrics,
        "pde_mse": pde_mse,
        "ic_mse": ic_mse,
        "bc_mse": bc_mse,
        "consistency_mse": cons_mse,
        "forward_time_ms": forward_time_ms,
        "T_pred": T_pred,
        "T_ref": T_ref_grid,
    }


In [ ]:
def extract_merlin_info(model):
    n_params = count_parameters(model)
    photons = getattr(model, 'n_photons', 0)
    depth = getattr(model, 'depth', 0)
    ql = None
    for m in model.modules():
        if hasattr(m, 'quantum_layer'):
            ql = m.quantum_layer
            break
    if ql is None and hasattr(model, 'quantum'):
        qmod = model.quantum
        if hasattr(qmod, 'circuit'):
            ql = qmod
        elif hasattr(qmod, 'quantum_layer'):
            ql = qmod.quantum_layer
    modes = 0
    fock_dim = 0
    bs = ps = ncomp = 0
    if ql is not None:
        modes = getattr(ql, 'n_modes', getattr(ql, 'input_size', 0))
        circ = getattr(ql, 'circuit', None)
        if circ is not None:
            try:
                ncomp = circ.ncomponents()
                desc = circ.describe()
                bs = desc.count('BS.')
                ps = desc.count('PS(')
                modes = getattr(circ, 'm', modes)
            except Exception:
                pass
        if modes > 0 and photons > 0:
            fock_dim = fock_dimension(modes, photons)
    return {"modes": modes, "photons": photons, "depth": depth,
            "fock_dim": fock_dim, "n_params": n_params,
            "bs": bs, "ps": ps, "ncomp": ncomp}

def run_experiment(model_factory, config, experiment_name, seed,
                   ablation_type="none", train=True, model=None,
                   print_every=None):
    set_seed(seed)
    row = {
        "model_name": experiment_name,
        "experiment_type": "ablation" if ablation_type != "none" else "baseline",
        "ablation_type": ablation_type,
        "seed": seed,
        "modes": 0,
        "photons": 0,
        "depth": 0,
        "fock_dim": 0,
        "n_params": 0,
        "ncomp": 0,
        "supported": True,
        "reason": "",
        "rmse": float('nan'),
        "nmse": float('nan'),
        "mae": float('nan'),
        "max_err": float('nan'),
        "pde_residual": float('nan'),
        "ic_error": float('nan'),
        "bc_error": float('nan'),
        "consistency_error": float('nan'),
        "train_time_s": 0.0,
        "forward_time_ms": 0.0,
    }
    try:
        if train:
            model = model_factory()
            for p in model.parameters():
                if p.is_floating_point():
                    p.data = p.data.to(DTYPE)
            print(f"\n>>> {experiment_name} | seed={seed} | epochs={config.epochs}")
            hist, elapsed = train_model(model, config, print_every=print_every)
            row["train_time_s"] = elapsed
        else:
            if model is None:
                raise ValueError("Model must be provided when train=False")
            print(f"\n>>> {experiment_name} | seed={seed} | evaluation only")
        info = extract_merlin_info(model)
        row.update({k: info[k] for k in ["modes", "photons", "depth", "fock_dim", "n_params", "ncomp"]})
        metrics = evaluate_model(model)
        row.update({
            "rmse": metrics["rmse"],
            "nmse": metrics["nmse"],
            "mae": metrics["mae"],
            "max_err": metrics["max_err"],
            "pde_residual": metrics["pde_mse"],
            "ic_error": metrics["ic_mse"],
            "bc_error": metrics["bc_mse"],
            "consistency_error": metrics["consistency_mse"],
            "forward_time_ms": metrics["forward_time_ms"],
        })
        print(f"    rmse={metrics['rmse']:.4e} | pde={metrics['pde_mse']:.4e} | "
              f"time={row['train_time_s']:.1f}s | fwd={metrics['forward_time_ms']:.2f}ms")
    except Exception as e:
        row["supported"] = False
        row["reason"] = str(e)
        print(f"    FAILED: {e}")
    return row, model


## 7. Quantum-Resource Ablations

We test which photonic resources matter using controlled ablations.

**Full baseline:** MerLin Aux QPINN with depth=1, 2 photons, 4 modes.


In [ ]:
all_rows = []
base_cfg = TrainConfig(epochs=CONFIG["base_epochs"], lr=CONFIG["lr"],
                       n_f=CONFIG["n_f"], n_i=CONFIG["n_i"], n_b=CONFIG["n_b"],
                       print_every=max(1, CONFIG["base_epochs"]//10))
pe = max(1, CONFIG["base_epochs"]//10)

# --- Baseline: Full MerLin ---
for seed in CONFIG["seeds"]:
    row, model_full = run_experiment(
        lambda: MerlinAuxQPINN(feature_size=CONFIG["feature_size"],
                               quantum_output_size=CONFIG["quantum_output_size"],
                               hidden=CONFIG["hidden"],
                               depth=CONFIG["baseline_depth"],
                               n_photons=CONFIG["baseline_photons"]),
        base_cfg, "MerLin Full", seed, ablation_type="none", print_every=pe)
    all_rows.append(row)

# --- Baseline: Classical matched ---
# Match parameter count roughly
for seed in CONFIG["seeds"]:
    row, model_classical = run_experiment(
        lambda: ClassicalAuxPINN(hidden=14, depth=2),
        base_cfg, "Classical Matched", seed, ablation_type="none", print_every=pe)
    all_rows.append(row)


# --- Strong classical surrogate: Fourier-feature PINN ---
for seed in CONFIG["seeds"]:
    row, _ = run_experiment(
        lambda: ClassicalFourierPINN(hidden=32, depth=3, n_fourier=16),
        base_cfg, "Classical Fourier", seed, ablation_type="none", print_every=pe)
    all_rows.append(row)


# --- Ablation 1: No mode mixing (retrained) ---
print("\n=== Ablation 1: No Mode Mixing ===")
for seed in CONFIG["seeds"]:
    row, _ = run_experiment(
        lambda: MerlinNoMixingAuxQPINN(feature_size=CONFIG["feature_size"],
                                       quantum_output_size=CONFIG["quantum_output_size"],
                                       hidden=CONFIG["hidden"],
                                       depth=1,
                                       n_photons=CONFIG["baseline_photons"]),
        base_cfg, "MerLin No-Mixing", seed, ablation_type="no_mixing_retrained", print_every=pe)
    all_rows.append(row)

# --- Ablation 2: Frozen quantum layer (surgical) ---
print("\n=== Ablation 2: Frozen Quantum Layer ===")
# Train full model, then freeze quantum and evaluate (no extra retraining)
set_seed(CONFIG["seeds"][0])
model_frozen = MerlinAuxQPINN(feature_size=CONFIG["feature_size"],
                              quantum_output_size=CONFIG["quantum_output_size"],
                              hidden=CONFIG["hidden"],
                              depth=CONFIG["baseline_depth"],
                              n_photons=CONFIG["baseline_photons"])
for p in model_frozen.parameters():
    if p.is_floating_point():
        p.data = p.data.to(DTYPE)
print(">>> Training full model before freezing...")
hist_full, elapsed_full = train_model(model_frozen, base_cfg, print_every=pe)
# Freeze quantum parameters
for name, param in model_frozen.named_parameters():
    if "quantum" in name or "post" in name:
        param.requires_grad = False
print(">>> Evaluating with frozen trained quantum layer...")
row_frozen, _ = run_experiment(None, base_cfg, "MerLin Frozen",
                               CONFIG["seeds"][0], ablation_type="frozen_surgical",
                               train=False, model=model_frozen)
row_frozen["train_time_s"] = elapsed_full
all_rows.append(row_frozen)

# Also: frozen random features (train from scratch with frozen quantum)
print("\n=== Ablation 2b: Frozen Random Quantum Features ===")
for seed in CONFIG["seeds"]:
    set_seed(seed)
    model_rand = MerlinAuxQPINN(feature_size=CONFIG["feature_size"],
                                quantum_output_size=CONFIG["quantum_output_size"],
                                hidden=CONFIG["hidden"],
                                depth=CONFIG["baseline_depth"],
                                n_photons=CONFIG["baseline_photons"])
    for p in model_rand.parameters():
        if p.is_floating_point():
            p.data = p.data.to(DTYPE)
    for name, param in model_rand.named_parameters():
        if "quantum" in name or "post" in name:
            param.requires_grad = False
    print(f">>> MerLin Frozen Random | seed={seed}")
    hist_rand, elapsed_rand = train_model(model_rand, base_cfg, print_every=pe)
    row_rand, _ = run_experiment(None, base_cfg, "MerLin Frozen Random",
                                 seed, ablation_type="frozen_random",
                                 train=False, model=model_rand)
    row_rand["train_time_s"] = elapsed_rand
    all_rows.append(row_rand)

# --- Ablation 3: Phase randomization (surgical, best effort) ---
print("\n=== Ablation 3: Randomized Phases ===")
# Try to randomize phase parameters in the trained full model
set_seed(CONFIG["seeds"][0])
model_phase = MerlinAuxQPINN(feature_size=CONFIG["feature_size"],
                             quantum_output_size=CONFIG["quantum_output_size"],
                             hidden=CONFIG["hidden"],
                             depth=CONFIG["baseline_depth"],
                             n_photons=CONFIG["baseline_photons"])
for p in model_phase.parameters():
    if p.is_floating_point():
        p.data = p.data.to(DTYPE)
print(">>> Training full model before phase randomization...")
hist_phase, _ = train_model(model_phase, base_cfg, print_every=pe)
phase_success = False
phase_reason = ""
try:
    ql = None
    for m in model_phase.modules():
        if hasattr(m, 'quantum_layer'):
            ql = m.quantum_layer
            break
    if ql is None and hasattr(model_phase, 'quantum'):
        qmod = model_phase.quantum
        if hasattr(qmod, 'quantum_layer'):
            ql = qmod.quantum_layer
    if ql is not None:
        circ = getattr(ql, 'circuit', None)
        if circ is not None:
            n_rand = 0
            for comp in circ._components:
                try:
                    comp_str = str(type(comp))
                    if 'PS' in comp_str or 'Phase' in comp_str:
                        for p in comp.get_parameters():
                            p.set_value(float(np.random.uniform(0, 2*math.pi)))
                            n_rand += 1
                except Exception:
                    pass
            if n_rand > 0:
                phase_success = True
                phase_reason = f"Randomized {n_rand} phase parameters"
            else:
                # Fallback: randomize all quantum layer torch parameters
                for name, param in model_phase.named_parameters():
                    if 'quantum' in name:
                        param.data.normal_(0, 0.5)
                        n_rand += 1
                phase_success = True
                phase_reason = f"Randomized {n_rand} quantum parameters (full parameter randomization fallback)"
        else:
            # simple() layer: randomize all quantum parameters
            n_rand = 0
            for name, param in model_phase.named_parameters():
                if 'quantum' in name:
                    param.data.normal_(0, 0.5)
                    n_rand += 1
            phase_success = True
            phase_reason = f"Randomized {n_rand} quantum parameters (simple layer fallback)"
    else:
        phase_reason = "Cannot access quantum layer"
except Exception as e:
    phase_reason = str(e)

if phase_success:
    row_phase, _ = run_experiment(None, base_cfg, "MerLin Phase-Random",
                                  CONFIG["seeds"][0], ablation_type="phase_random_surgical",
                                  train=False, model=model_phase)
    all_rows.append(row_phase)
else:
    all_rows.append({
        "model_name": "MerLin Phase-Random", "experiment_type": "ablation",
        "ablation_type": "phase_random_surgical", "seed": CONFIG["seeds"][0],
        "modes": 0, "photons": 0, "depth": 0, "fock_dim": 0,
        "n_params": 0, "ncomp": 0, "supported": False, "reason": phase_reason,
        "rmse": float('nan'), "nmse": float('nan'), "mae": float('nan'),
        "max_err": float('nan'), "pde_residual": float('nan'),
        "ic_error": float('nan'), "bc_error": float('nan'),
        "consistency_error": float('nan'), "train_time_s": 0.0, "forward_time_ms": 0.0,
    })
    print(f"    Phase randomization unsupported: {phase_reason}")

# --- Ablation 4: Depth reduction ---
print("\n=== Ablation 4: Depth Reduction ===")
depth_vals = CONFIG["depth_values_fast"] if CONFIG["FAST_MODE"] else CONFIG["depth_values_full"]
for d in depth_vals:
    for seed in CONFIG["seeds"]:
        row, _ = run_experiment(
            lambda d=d: MerlinAuxQPINN(feature_size=CONFIG["feature_size"],
                                       quantum_output_size=CONFIG["quantum_output_size"],
                                       hidden=CONFIG["hidden"],
                                       depth=d,
                                       n_photons=CONFIG["baseline_photons"]),
            base_cfg, f"MerLin Depth={d}", seed, ablation_type=f"depth_{d}", print_every=pe)
        all_rows.append(row)

# --- Ablation 5: Distinguishability (unsupported / TODO) ---
print("\n=== Ablation 5: Distinguishability ===")
print("TODO: Perceval supports pcvl.Source(indistinguishability=mu), but")
print("surgical mixing of distinguishable vs indistinguishable probabilities")
print("requires rebuilding the experiment for each mu. Retrained sweeps are")
print("possible with custom CircuitBuilder experiments, but not implemented")
print("in this FAST_MODE pass to avoid excessive runtime.")
for mu in CONFIG["mu_values"]:
    all_rows.append({
        "model_name": f"MerLin Distinguishability mu={mu}",
        "experiment_type": "ablation", "ablation_type": "distinguishability",
        "seed": CONFIG["seeds"][0], "modes": 0, "photons": 0, "depth": 0,
        "fock_dim": 0, "n_params": 0, "ncomp": 0,
        "supported": False,
        "reason": "Distinguishability sweep requires custom experiment rebuilding per mu value. Not implemented in this pass.",
        "rmse": float('nan'), "nmse": float('nan'), "mae": float('nan'),
        "max_err": float('nan'), "pde_residual": float('nan'),
        "ic_error": float('nan'), "bc_error": float('nan'),
        "consistency_error": float('nan'), "train_time_s": 0.0, "forward_time_ms": 0.0,
    })

# --- Ablation 6: Photon-number sweep ---
print("\n=== Ablation 6: Photon Sweep ===")
photon_vals = CONFIG["photon_values_fast"] if CONFIG["FAST_MODE"] else CONFIG["photon_values_full"]
for n_ph in photon_vals:
    for seed in CONFIG["seeds"]:
        row, _ = run_experiment(
            lambda n_ph=n_ph: MerlinAuxQPINN(feature_size=CONFIG["feature_size"],
                                             quantum_output_size=CONFIG["quantum_output_size"],
                                             hidden=CONFIG["hidden"],
                                             depth=CONFIG["baseline_depth"],
                                             n_photons=n_ph),
            base_cfg, f"MerLin Photons={n_ph}", seed, ablation_type=f"photon_{n_ph}", print_every=pe)
        all_rows.append(row)


## 8. Classical-Simulability Scaling

Controlled sweeps to measure whether simulation cost grows with photonic resources.


In [ ]:
# --- Scaling Sweep A: Depth ---
print("\n=== Scaling A: Depth ===")
scaling_depths = [1, 2, 3] if CONFIG["FAST_MODE"] else [1, 2, 3, 4, 5]
for d in scaling_depths:
    for seed in CONFIG["seeds"]:
        row, _ = run_experiment(
            lambda d=d: MerlinAuxQPINN(feature_size=4, quantum_output_size=4,
                                       hidden=16, depth=d, n_photons=2),
            TrainConfig(epochs=CONFIG["sweep_epochs"]),
            f"Scale Depth={d}", seed, ablation_type="scale_depth", print_every=0)
        all_rows.append(row)

# --- Scaling Sweep B: Modes ---
print("\n=== Scaling B: Modes ===")
scaling_modes = [2, 3, 4] if CONFIG["FAST_MODE"] else [2, 3, 4, 5, 6, 8]
for m in scaling_modes:
    feature_size = max(2, m - 1)
    for seed in CONFIG["seeds"]:
        row, _ = run_experiment(
            lambda m=m: MerlinAuxQPINN(feature_size=feature_size, quantum_output_size=4,
                                       hidden=16, depth=2, n_photons=2),
            TrainConfig(epochs=CONFIG["sweep_epochs"]),
            f"Scale Modes={m}", seed, ablation_type="scale_modes", print_every=0)
        all_rows.append(row)

# --- Scaling Sweep C: Photons ---
print("\n=== Scaling C: Photons ===")
scaling_photons = [1, 2] if CONFIG["FAST_MODE"] else [1, 2, 3]
for n_ph in scaling_photons:
    for seed in CONFIG["seeds"]:
        row, _ = run_experiment(
            lambda n_ph=n_ph: MerlinAuxQPINN(feature_size=4, quantum_output_size=4,
                                             hidden=16, depth=2, n_photons=n_ph),
            TrainConfig(epochs=CONFIG["sweep_epochs"]),
            f"Scale Photons={n_ph}", seed, ablation_type="scale_photons", print_every=0)
        all_rows.append(row)


In [ ]:
df = pd.DataFrame(all_rows)
# Drop internal tensor arrays if any
for col in ["T_pred", "T_ref"]:
    if col in df.columns:
        df = df.drop(columns=[col])

csv_path = os.path.join(CONFIG["output_dir"], "quantum_resource_ablation_scaling_results.csv")
df.to_csv(csv_path, index=False)
print(f"Saved {len(df)} rows to {csv_path}")
print("\n=== Main Results ===")
display_cols = ["model_name", "ablation_type", "supported", "modes", "photons", "depth",
                "fock_dim", "n_params", "ncomp", "rmse", "pde_residual", "train_time_s", "forward_time_ms"]
print(df[display_cols].to_string(index=False))


In [ ]:
# Compute relative RMSE degradation for supported ablations
baseline_rows = df[(df["ablation_type"] == "none") & (df["model_name"] == "MerLin Full")]
if len(baseline_rows) > 0:
    baseline_rmse = baseline_rows["rmse"].mean()
else:
    baseline_rmse = 1.0

ablation_names = []
ablation_deltas = []
ablation_colors = []

# Map ablation types to labels
ablation_map = {
    "no_mixing_retrained": "No Mixing",
    "frozen_surgical": "Frozen (trained)",
    "frozen_random": "Frozen (random)",
    "phase_random_surgical": "Phase Random",
    "depth_0": "Depth 0 (classical)",
    "photon_1": "1 Photon",
}

for abl_type, label in ablation_map.items():
    sub = df[df["ablation_type"] == abl_type]
    sub = sub[sub["supported"] == True]
    if len(sub) > 0 and baseline_rmse > 0:
        delta = (sub["rmse"].mean() - baseline_rmse) / baseline_rmse
        ablation_names.append(label)
        ablation_deltas.append(delta)
        ablation_colors.append("#d62728" if delta > 0 else "#2ca02c")

if len(ablation_names) > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(range(len(ablation_names)), ablation_deltas, color=ablation_colors, edgecolor="black")
    ax.set_xticks(range(len(ablation_names)))
    ax.set_xticklabels(ablation_names, rotation=30, ha="right")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_ylabel("Relative RMSE Degradation")
    ax.set_title(f"Ablation Impact (baseline RMSE={baseline_rmse:.4e})")
    ax.grid(True, axis="y", ls="--", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["output_dir"], "figures", "ablation_degradation.png"), dpi=150)
    plt.show()
else:
    print("No supported ablations to plot.")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Depth scaling
sub_depth = df[df["ablation_type"] == "scale_depth"]
sub_depth = sub_depth[sub_depth["supported"] == True]
if len(sub_depth) > 0:
    ax = axes[0]
    for _, grp in sub_depth.groupby("depth"):
        ax.scatter(grp["fock_dim"].mean(), grp["forward_time_ms"].mean(),
                   s=100, label=f"depth={grp['depth'].iloc[0]}")
    ax.set_xlabel("Fock Dimension")
    ax.set_ylabel("Forward Time (ms)")
    ax.set_title("Depth Scaling")
    ax.set_yscale("log")
    ax.grid(True, ls="--", alpha=0.3)

# Mode scaling
sub_modes = df[df["ablation_type"] == "scale_modes"]
sub_modes = sub_modes[sub_modes["supported"] == True]
if len(sub_modes) > 0:
    ax = axes[1]
    for _, grp in sub_modes.groupby("modes"):
        ax.scatter(grp["fock_dim"].mean(), grp["forward_time_ms"].mean(),
                   s=100, label=f"modes={grp['modes'].iloc[0]}")
    ax.set_xlabel("Fock Dimension")
    ax.set_ylabel("Forward Time (ms)")
    ax.set_title("Mode Scaling")
    ax.set_yscale("log")
    ax.grid(True, ls="--", alpha=0.3)

# Photon scaling
sub_ph = df[df["ablation_type"] == "scale_photons"]
sub_ph = sub_ph[sub_ph["supported"] == True]
if len(sub_ph) > 0:
    ax = axes[2]
    for _, grp in sub_ph.groupby("photons"):
        ax.scatter(grp["fock_dim"].mean(), grp["forward_time_ms"].mean(),
                   s=100, label=f"photons={grp['photons'].iloc[0]}")
    ax.set_xlabel("Fock Dimension")
    ax.set_ylabel("Forward Time (ms)")
    ax.set_title("Photon Scaling")
    ax.set_yscale("log")
    ax.grid(True, ls="--", alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG["output_dir"], "figures", "scaling_plot.png"), dpi=150)
plt.show()


## 10. Final Verdict Table

Auto-generated from experimental results. Conservative language only.


In [ ]:
def verdict_for(question, method, evidence, verdict):
    return {"Question": question, "Method": method, "Evidence": evidence, "Verdict": verdict}

verdicts = []

# 1. Does mode mixing matter?
sub = df[df["ablation_type"] == "no_mixing_retrained"]
sub = sub[sub["supported"] == True]
if len(sub) > 0 and baseline_rmse > 0:
    delta = (sub["rmse"].mean() - baseline_rmse) / baseline_rmse
    if delta > 0.5:
        v1 = verdict_for("Does mode mixing matter?", "No-mixing retrained ablation",
                         f"RMSE degraded by {delta:.1%} without beam splitters", "Supported")
    elif delta > 0.1:
        v1 = verdict_for("Does mode mixing matter?", "No-mixing retrained ablation",
                         f"RMSE degraded by {delta:.1%}", "Partially supported")
    else:
        v1 = verdict_for("Does mode mixing matter?", "No-mixing retrained ablation",
                         f"RMSE change {delta:.1%}", "Not supported")
else:
    v1 = verdict_for("Does mode mixing matter?", "No-mixing retrained ablation",
                     "Experiment failed or unsupported", "Inconclusive")
verdicts.append(v1)

# 2. Does training the quantum layer matter?
sub_frozen = df[df["ablation_type"].isin(["frozen_surgical", "frozen_random"])]
sub_frozen = sub_frozen[sub_frozen["supported"] == True]
if len(sub_frozen) > 0:
    frozen_rmse = sub_frozen["rmse"].mean()
    delta = (frozen_rmse - baseline_rmse) / baseline_rmse
    if delta > 0.3:
        v2 = verdict_for("Does training the quantum layer matter?", "Frozen quantum + retrained readout",
                         f"Frozen RMSE {frozen_rmse:.4e} vs baseline {baseline_rmse:.4e}", "Supported")
    elif delta > 0.05:
        v2 = verdict_for("Does training the quantum layer matter?", "Frozen quantum + retrained readout",
                         f"Frozen RMSE {frozen_rmse:.4e} vs baseline {baseline_rmse:.4e}", "Partially supported")
    else:
        v2 = verdict_for("Does training the quantum layer matter?", "Frozen quantum + retrained readout",
                         f"Frozen RMSE {frozen_rmse:.4e} vs baseline {baseline_rmse:.4e}", "Not supported")
else:
    v2 = verdict_for("Does training the quantum layer matter?", "Frozen quantum + retrained readout",
                     "No supported frozen experiments", "Inconclusive")
verdicts.append(v2)

# 3. Do learned optical phases matter?
sub_phase = df[df["ablation_type"] == "phase_random_surgical"]
sub_phase = sub_phase[sub_phase["supported"] == True]
if len(sub_phase) > 0:
    phase_rmse = sub_phase["rmse"].mean()
    delta = (phase_rmse - baseline_rmse) / baseline_rmse
    if delta > 0.3:
        v3 = verdict_for("Do learned optical phases matter?", "Phase randomization after training",
                         f"Randomized-phase RMSE {phase_rmse:.4e} vs baseline {baseline_rmse:.4e}", "Supported")
    elif delta > 0.05:
        v3 = verdict_for("Do learned optical phases matter?", "Phase randomization after training",
                         f"Randomized-phase RMSE {phase_rmse:.4e} vs baseline {baseline_rmse:.4e}", "Partially supported")
    else:
        v3 = verdict_for("Do learned optical phases matter?", "Phase randomization after training",
                         f"Randomized-phase RMSE {phase_rmse:.4e} vs baseline {baseline_rmse:.4e}", "Not supported")
else:
    v3 = verdict_for("Do learned optical phases matter?", "Phase randomization after training",
                     "Phase parameters not accessible or unsupported", "Unsupported by current API/runtime")
verdicts.append(v3)

# 4. Does multiphoton interference matter?
sub_mu = df[df["ablation_type"] == "distinguishability"]
if len(sub_mu) > 0 and any(sub_mu["supported"]):
    # Would analyze mu sweep here
    v4 = verdict_for("Does multiphoton interference matter?", "Distinguishability sweep",
                     "Partial results available", "Inconclusive")
else:
    v4 = verdict_for("Does multiphoton interference matter?", "Distinguishability sweep",
                     "Not implemented in this pass", "Unsupported by current API/runtime")
verdicts.append(v4)

# 5. Does photon number matter?
sub_ph = df[df["ablation_type"].str.startswith("photon_")]
sub_ph = sub_ph[sub_ph["supported"] == True]
if len(sub_ph) > 1:
    ph_min = sub_ph.loc[sub_ph["photons"].idxmin()]
    ph_max = sub_ph.loc[sub_ph["photons"].idxmax()]
    if ph_max["rmse"] < ph_min["rmse"] * 0.9:
        v5 = verdict_for("Does photon number matter?", "Photon sweep",
                         f"RMSE improved from {ph_min['rmse']:.4e} to {ph_max['rmse']:.4e}", "Supported")
    else:
        v5 = verdict_for("Does photon number matter?", "Photon sweep",
                         f"RMSE {ph_min['rmse']:.4e} vs {ph_max['rmse']:.4e}", "Not supported")
else:
    v5 = verdict_for("Does photon number matter?", "Photon sweep",
                     "Insufficient data", "Inconclusive")
verdicts.append(v5)

# 6. Does increasing depth help?
sub_depth = df[df["ablation_type"].str.startswith("depth_")]
sub_depth = sub_depth[sub_depth["supported"] == True]
if len(sub_depth) > 1:
    d_min = sub_depth.loc[sub_depth["depth"].idxmin()]
    d_max = sub_depth.loc[sub_depth["depth"].idxmax()]
    if d_max["rmse"] < d_min["rmse"] * 0.9:
        v6 = verdict_for("Does increasing depth help?", "Depth sweep",
                         f"RMSE improved from {d_min['rmse']:.4e} to {d_max['rmse']:.4e}", "Supported")
    else:
        v6 = verdict_for("Does increasing depth help?", "Depth sweep",
                         f"RMSE {d_min['rmse']:.4e} vs {d_max['rmse']:.4e}", "Not supported")
else:
    v6 = verdict_for("Does increasing depth help?", "Depth sweep",
                     "Insufficient data", "Inconclusive")
verdicts.append(v6)

# 7. Does classical simulation cost grow?
sub_scale = df[df["ablation_type"].isin(["scale_depth", "scale_modes", "scale_photons"])]
sub_scale = sub_scale[sub_scale["supported"] == True]
if len(sub_scale) > 1:
    corr = sub_scale["fock_dim"].corr(sub_scale["forward_time_ms"])
    if corr > 0.5:
        v7 = verdict_for("Does classical simulation cost grow with modes/photons/depth?", "Scaling sweeps",
                         f"Forward time correlates with Fock dim (r={corr:.2f})", "Supported")
    else:
        v7 = verdict_for("Does classical simulation cost grow with modes/photons/depth?", "Scaling sweeps",
                         f"Correlation r={corr:.2f}", "Partially supported")
else:
    v7 = verdict_for("Does classical simulation cost grow with modes/photons/depth?", "Scaling sweeps",
                     "Insufficient data", "Inconclusive")
verdicts.append(v7)

# 8. Is the current model plausibly quantum meaningful?
# Aggregate assessment
supported_count = sum(1 for v in verdicts if v["Verdict"] in ["Supported", "Partially supported"])
if supported_count >= 4:
    v8 = verdict_for("Is the current model plausibly quantum meaningful?", "Aggregate assessment",
                     f"{supported_count}/7 ablations show quantum-resource sensitivity. Quantum advantage: not claimed.", "Supported")
elif supported_count >= 2:
    v8 = verdict_for("Is the current model plausibly quantum meaningful?", "Aggregate assessment",
                     f"{supported_count}/7 ablations show sensitivity). Quantum advantage: not claimed.", "Partially supported")
else:
    v8 = verdict_for("Is the current model plausibly quantum meaningful?", "Aggregate assessment",
                     f"{supported_count}/7 ablations show sensitivity). Quantum advantage: not claimed.", "Not supported")
verdicts.append(v8)

df_verdict = pd.DataFrame(verdicts)
print("\n=== Final Verdict Table ===")
print(df_verdict.to_string(index=False))

# Save verdict CSV
df_verdict.to_csv(os.path.join(CONFIG["output_dir"], "final_verdict.csv"), index=False)
